<a href="https://colab.research.google.com/github/MBR4V0/Python/blob/main/Game_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from IPython.display import HTML, display

html = r'''
<style>
#wrap{max-width:720px;margin:auto;text-align:center;font-family:-apple-system,sans-serif;user-select:none;-webkit-user-select:none;-webkit-touch-callout:none}
#g{width:100%;display:block;border:3px solid #333;border-radius:10px;background:#87ceeb;touch-action:none;box-sizing:border-box}
#ctl{display:flex;justify-content:space-between;margin-top:10px;touch-action:none}
.grp{display:flex;gap:12px}
.b{width:72px;height:72px;border-radius:50%;border:none;background:#333;color:#fff;font-size:26px;font-weight:bold;touch-action:none;-webkit-tap-highlight-color:transparent;opacity:.85}
.b.on{background:#e63946}
.b.small{font-size:14px}
#info{margin:8px;color:#555;font-size:13px}
</style>
<div id="wrap">
  <canvas id="g" width="720" height="480"></canvas>
  <div id="ctl">
    <div class="grp"><button class="b" data-k="ArrowLeft">&#9664;</button><button class="b" data-k="ArrowRight">&#9654;</button></div>
    <div class="grp"><button class="b small" data-k="KeyE">AÇÃO</button><button class="b" data-k="Space">&#8679;</button></div>
  </div>
  <p id="info">Teclado: ← → / A D mover | ↑ / W / ESPAÇO pular | E = ação | ENTER reinicia</p>
</div>
<script>
(function(){
const cv=document.getElementById('g'),ctx=cv.getContext('2d');
const T=32,W=720,H=480,ROWS=15;
let keys={},state='title',level=1,lives=3,score=0,coinsGot=0,frame=0,timer=0,overT=0;
let grid,cols,enemies,bats,spikes,coins,keyItems,safes,decor,eggs,particles,cp,gate;
let p,camX=0,jumpHeld=false,coyote=0,buf=0,prevAct=false,cpX=64;
let villain=null,vTimer=0,eggTimer=0;
let riding=false,ponyRef=null,sitting=false,rainbow=false,crown=0;
let toastMsg="",toastT=0,msgCool=0,audio=null,tapN=0,tapT=0,konami=[];
const KON=['ArrowUp','ArrowUp','ArrowDown','ArrowDown','ArrowLeft','ArrowRight','ArrowLeft','ArrowRight','KeyB','KeyA'];
const KC={gold:'#ffd700',blue:'#3aa0ff',silver:'#e0e0e0'};
const KN={gold:'dourada',blue:'azul',silver:'prateada'};
const VNAMES=['Barão Corvo','Capitão Sombra','Dr. Quackenstein','Rei Pombo-Gigante','O Pato Negro'];
const SIGNS=['Placa inútil. Parabéns por lê-la.','Atenção: esta placa não avisa nada.','Você está aqui. Sim, aqui mesmo.','Não alimente o programador.','Cuidado com o chão. E com o céu.'];
const LEVERS=['Nada aconteceu.','Você sentiu um vazio interior.','Clique. Só isso.','Em algum lugar, uma lâmpada piscou. Talvez.'];
const GIFTS=['Era só uma meia.','Dentro havia um bilhete: "continue".','Um patinho de borracha! Inútil, mas lindo.','Uma caixa vazia. O pensamento é que conta.'];

function initAudio(){
  if(!audio){try{audio=new (window.AudioContext||window.webkitAudioContext)();}catch(_){}}
  if(audio&&audio.state==='suspended')audio.resume();
}
function beep(f,d,type){
  if(!audio)return;
  try{
    const o=audio.createOscillator(),g=audio.createGain();
    o.type=type||'square';o.frequency.value=f;g.gain.value=0.05;
    o.connect(g);g.connect(audio.destination);
    o.start();g.gain.exponentialRampToValueAtTime(0.0001,audio.currentTime+d);
    o.stop(audio.currentTime+d);
  }catch(_){}
}
function toast(m,t){toastMsg=m;toastT=t||150;}
function rng(seed){return function(){seed|=0;seed=seed+0x6D2B79F5|0;let t=Math.imul(seed^seed>>>15,1|seed);t=t+Math.imul(t^t>>>7,61|t)^t;return((t^t>>>14)>>>0)/4294967296;}}
function hit(a,b){return a.x<b.x+b.w&&a.x+a.w>b.x&&a.y<b.y+b.h&&a.y+a.h>b.y;}
function startIfTitle(){if(state==='title')newGame();}

addEventListener('keydown',e=>{
  keys[e.code]=true;initAudio();
  if(['ArrowUp','ArrowDown','ArrowLeft','ArrowRight','Space'].includes(e.code))e.preventDefault();
  if(!e.repeat){
    konami.push(e.code);if(konami.length>10)konami.shift();
    if(konami.join()===KON.join()){rainbow=!rainbow;toast('CÓDIGO SECRETO! Pato arco-íris '+(rainbow?'ativado':'desativado')+'!',200);}
  }
  startIfTitle();
});
addEventListener('keyup',e=>{keys[e.code]=false;});

document.querySelectorAll('.b').forEach(b=>{
  const k=b.dataset.k;
  const on=e=>{e.preventDefault();keys[k]=true;b.classList.add('on');initAudio();startIfTitle();};
  const off=e=>{e.preventDefault();keys[k]=false;b.classList.remove('on');};
  b.addEventListener('pointerdown',on);
  ['pointerup','pointercancel','pointerleave'].forEach(ev=>b.addEventListener(ev,off));
  b.addEventListener('contextmenu',e=>e.preventDefault());
});

cv.addEventListener('pointerdown',e=>{
  e.preventDefault();initAudio();
  if(state==='title'){newGame();return;}
  if((state==='over'&&overT<=0)||state==='win'){newGame();return;}
  if(!p)return;
  const r=cv.getBoundingClientRect(),x=(e.clientX-r.left)*W/r.width,y=(e.clientY-r.top)*H/r.height;
  if(Math.abs(x-(p.x-camX+p.w/2))<36&&Math.abs(y-(p.y+p.h/2))<42){
    tapN++;tapT=45;beep(800,0.08,'triangle');
    if(tapN>=5){tapN=0;crown=900;toast('QUACK! Todos saúdem o Rei Pato!',200);beep(500,0.3);}
  }
});

// ---------- FASE ----------
function loadLevel(){
  const r=rng(1234+level*77);
  cols=125+level*15;
  grid=Array.from({length:ROWS},()=>Array(cols+2).fill(0));
  enemies=[];bats=[];spikes=[];coins=[];keyItems=[];safes=[];decor=[];eggs=[];particles=[];
  const gap=new Array(cols+10).fill(false),used=new Array(cols+10).fill(false);
  for(let i=0;i<10;i++)used[i]=true;
  let x=12;
  while(x<cols-18){
    if(r()<0.7){const w=2+Math.floor(r()*(level>2?3:2));for(let i=0;i<w;i++)gap[x+i]=true;x+=w;}
    x+=5+Math.floor(r()*5);
  }
  for(let c=0;c<cols;c++)if(!gap[c]){grid[13][c]=1;grid[14][c]=3;}
  const plats=[];
  let c=10;
  while(c<cols-20){
    const len=3+Math.floor(r()*3),row=10+Math.floor(r()*2);
    for(let i=0;i<len;i++)grid[row][c+i]=2;
    plats.push({c:c,row:row,len:len});
    for(let i=0;i<len;i++)if(r()<0.6)coins.push({x:(c+i)*T+8,y:(row-1)*T+6,got:false});
    c+=len+4+Math.floor(r()*6);
  }
  for(let c=8;c<cols-24;c++){
    let ok=true;
    for(let d=-2;d<=2;d++)if(gap[c+d]||grid[10][c+d]||grid[11][c+d]||grid[12][c+d])ok=false;
    if(ok&&r()<0.07){const h=1+Math.floor(r()*2);for(let i=0;i<h;i++)grid[12-i][c]=2;c+=3;}
  }
  const spot=(c)=>{
    for(;c<cols-20;c++){
      if(!used[c]&&grid[13][c-1]&&grid[13][c]&&grid[13][c+1]&&!grid[12][c-1]&&!grid[12][c]&&!grid[12][c+1]&&!grid[11][c]){
        for(let d=-2;d<=2;d++)used[c+d]=true;return c;
      }
    }
    return 14;
  };
  // chaves e cofres
  const mk=(id,pl)=>keyItems.push({id:id,x:(pl.c+Math.floor(pl.len/2))*T+8,y:pl.row*T-20,w:16,h:16,state:'world',fall:false,vy:0});
  mk('gold',plats[2]);
  mk('blue',plats[Math.floor(plats.length*0.65)]);
  const cg=spot(Math.floor(cols*0.72)),cb=spot(22),cc=spot(Math.floor(cols*0.45));
  safes.push({kind:'gold',x:cg*T+1,y:13*T-30,w:30,h:30,open:false});
  safes.push({kind:'blue',x:cb*T+1,y:13*T-30,w:30,h:30,open:false});
  cp={x:cc*T,reached:false};cpX=2*T;
  // decoração sem utilidade
  ponyRef={kind:'pony',x:6*T,y:13*T-28,w:40,h:28,ridden:false};
  decor.push(ponyRef);
  const c1=spot(Math.floor(cols*0.3));decor.push({kind:'bench',x:c1*T-6,y:13*T-20,w:44,h:20});
  const c2=spot(Math.floor(cols*0.58));decor.push({kind:'lever',x:c2*T+6,y:13*T-30,w:20,h:30,on:false});
  const c3=spot(Math.floor(cols*0.85));decor.push({kind:'sign',x:c3*T,y:13*T-36,w:30,h:36,text:SIGNS[level%SIGNS.length]});
  // espinhos
  const sp=0.05+level*0.01;
  for(let c=14;c<cols-22;c++){
    if(!used[c]&&grid[13][c]&&grid[13][c-1]&&grid[13][c+1]&&!grid[12][c]&&!grid[12][c-1]&&!grid[12][c+1]&&r()<sp){
      spikes.push({x:c*T+3,y:13*T-14,w:T-6,h:14});
      for(let d=-1;d<=1;d++)used[c+d]=true;
    }
  }
  // moedas no chão
  for(let c=14;c<cols-12;c++)if(grid[13][c]===1&&!grid[12][c]&&!used[c]&&r()<0.05)coins.push({x:c*T+8,y:11*T+6,got:false});
  // inimigos terrestres
  const prob=Math.min(0.14,0.06+level*0.015);
  for(let c=16;c<cols-24;c++){
    if(grid[13][c]===1&&!grid[12][c]&&!grid[12][c-1]&&!grid[12][c+1]&&r()<prob){
      enemies.push({x:c*T+3,y:13*T-24,w:26,h:24,vx:0,vy:0,dir:r()<0.5?-1:1,dead:false,t:0,onGround:false});
      c+=2;
    }
  }
  // morcegos
  for(let c=26;c<cols-24;c+=Math.max(8,22-level*3)+Math.floor(r()*8)){
    bats.push({bx:c*T,x:c*T,y0:280+r()*40,y:300,ph:r()*6,w:28,h:20,dead:false,t:0});
  }
  // portão
  gate={col:cols-9,open:false};
  for(let rr=8;rr<=12;rr++)grid[rr][gate.col]=4;
  p={x:2*T,y:13*T-29,w:24,h:28,vx:0,vy:0,dir:1,onGround:false,inv:60};
  camX=0;state='play';villain=null;riding=false;sitting=false;
  vTimer=700+Math.floor(Math.random()*1400);
  eggTimer=300+Math.floor(Math.random()*400);
  toast('Fase '+level+': o portão está trancado. Ache a chave certa!',260);
}
function newGame(){level=1;lives=3;score=0;coinsGot=0;loadLevel();}

// ---------- FÍSICA ----------
function solid(tx,ty){
  if(tx<0||tx>=cols)return true;
  if(ty<0||ty>=ROWS)return false;
  return grid[ty][tx]>0;
}
function moveX(e){
  e.x+=e.vx;
  const y0=Math.floor(e.y/T),y1=Math.floor((e.y+e.h-0.01)/T);
  const x0=Math.floor(e.x/T),x1=Math.floor((e.x+e.w-0.01)/T);
  for(let ty=y0;ty<=y1;ty++)for(let tx=x0;tx<=x1;tx++){
    if(solid(tx,ty)){
      if(e.vx>0)e.x=tx*T-e.w;else if(e.vx<0)e.x=(tx+1)*T;
      return true;
    }
  }
  return false;
}
function moveY(e){
  e.y+=e.vy;e.onGround=false;
  const y0=Math.floor(e.y/T),y1=Math.floor((e.y+e.h-0.01)/T);
  const x0=Math.floor(e.x/T),x1=Math.floor((e.x+e.w-0.01)/T);
  for(let ty=y0;ty<=y1;ty++)for(let tx=x0;tx<=x1;tx++){
    if(solid(tx,ty)){
      if(e.vy>0){e.y=ty*T-e.h;e.onGround=true;}else if(e.vy<0){e.y=(ty+1)*T;}
      e.vy=0;return;
    }
  }
}
function burst(x,y,n){
  for(let i=0;i<n;i++)particles.push({x:x,y:y,vx:(Math.random()-0.5)*8,vy:-Math.random()*7,life:50,c:['#ffd700','#fff','#ff8c1a'][i%3]});
}
function dismount(){
  if(!riding)return;
  riding=false;ponyRef.ridden=false;ponyRef.x=p.x-4;ponyRef.y=p.y+p.h-ponyRef.h;
}
function die(){
  lives--;beep(150,0.4,'sawtooth');
  if(lives<=0){state='over';overT=90;return;}
  p.x=cpX;p.y=13*T-29;p.vx=0;p.vy=0;p.inv=150;sitting=false;
  dismount();
  toast('Ops! Você perdeu uma vida.',120);
}
function hasKey(id){return keyItems.some(k=>k.id===id&&k.state==='inv');}
function nearDecor(){
  let b=null,bd=50;
  for(const d of decor){
    if(d.ridden)continue;
    const dd=Math.abs(d.x+d.w/2-(p.x+p.w/2));
    if(dd<bd&&Math.abs((d.y+d.h)-(p.y+p.h))<30){bd=dd;b=d;}
  }
  return b;
}
function doAction(){
  if(state!=='play')return;
  if(riding){if(p.onGround){dismount();toast('Você desceu do pônei. Ele não se importou.',140);}return;}
  if(sitting){sitting=false;return;}
  const d=nearDecor();
  if(!d){beep(600,0.08,'triangle');toast('Quack!',50);return;}
  if(d.kind==='pony'){riding=true;d.ridden=true;toast('Você montou no pônei! Ele não vai mais rápido. Nem mais devagar.',200);beep(400,0.15);}
  else if(d.kind==='bench'){sitting=true;p.vx=0;toast('Você sentou. Nada acontece. É bom assim.',180);}
  else if(d.kind==='lever'){d.on=!d.on;beep(300,0.1);toast(LEVERS[Math.floor(Math.random()*LEVERS.length)],160);}
  else if(d.kind==='sign'){toast(d.text,200);}
}

function updatePlayer(){
  const L=keys.ArrowLeft||keys.KeyA,R=keys.ArrowRight||keys.KeyD;
  const J=keys.ArrowUp||keys.KeyW||keys.Space;
  const A=keys.KeyE||keys.KeyZ;
  if(A&&!prevAct)doAction();
  prevAct=!!A;
  if(sitting&&(L||R||J))sitting=false;
  if(!sitting){
    if(L){p.vx-=0.5;p.dir=-1;}
    if(R){p.vx+=0.5;p.dir=1;}
    if(!L&&!R)p.vx*=0.8;
    p.vx=Math.max(-4.5,Math.min(4.5,p.vx));
    if(Math.abs(p.vx)<0.05)p.vx=0;
  }else p.vx=0;
  if(p.onGround)coyote=6;else coyote--;
  if(J&&!jumpHeld&&!sitting)buf=6;else buf--;
  if(buf>0&&coyote>0&&!sitting){p.vy=-12;coyote=0;buf=0;beep(520,0.12);}
  if(!J&&p.vy<-4)p.vy=-4;
  jumpHeld=!!J;
  p.vy=Math.min(14,p.vy+0.6);
  if(moveX(p))p.vx=0;
  moveY(p);
  if(p.x<0)p.x=0;
  if(p.inv>0)p.inv--;
  if(p.y>H+80)die();
}
function touchEnemy(e){
  if(e.dead||!hit(p,e))return;
  if(p.vy>0&&(p.y+p.h-e.y)<20){e.dead=true;e.t=20;p.vy=-9;score+=100;beep(700,0.15);}
  else if(p.inv<=0)die();
}

// ---------- VILÃO ----------
function spawnVillain(){
  villain={x:camX+W+40,y:-60,w:44,h:40,hp:3,mode:'hover',t:0,sx:0,sy:0,dir:-1,carry:null,flash:0,
    name:VNAMES[Math.floor(Math.random()*VNAMES.length)]};
  toast('DO NADA... apareceu '+villain.name+'!',230);
  beep(120,0.6,'sawtooth');
}
function updateVillain(){
  const v=villain;if(!v)return;
  if(v.flash>0)v.flash--;
  v.t++;
  const px=p.x+p.w/2,py=p.y+p.h/2;
  if(v.mode==='hover'){
    const tx=px+Math.sin(v.t/40)*160,ty=70+Math.sin(v.t/25)*20;
    v.x+=(tx-v.w/2-v.x)*0.03;v.y+=(ty-v.y)*0.05;
    v.dir=px>v.x?1:-1;
    if(v.t>150){v.mode='swoop';v.t=0;v.sx=px;v.sy=py;}
  }else if(v.mode==='swoop'){
    const dx=v.sx-(v.x+v.w/2),dy=v.sy-(v.y+v.h/2),d=Math.hypot(dx,dy)||1,s=4.5+level*0.3;
    v.x+=dx/d*s;v.y+=dy/d*s;
    if(d<12||v.t>90){v.mode='rise';v.t=0;}
  }else{
    v.y-=3;
    if(v.y<80||v.t>60){v.mode='hover';v.t=0;}
  }
  if(hit(p,v)){
    if(p.vy>0&&(p.y+p.h)-v.y<24&&v.flash===0){
      v.hp--;v.flash=60;p.vy=-11;score+=200;beep(300,0.2);v.mode='rise';v.t=0;
      if(v.hp<=0){
        if(v.carry){v.carry.state='world';v.carry.x=v.x;v.carry.y=v.y;v.carry.fall=true;v.carry.vy=0;}
        burst(v.x+22,v.y+20,40);score+=1000;
        toast(v.name+' foi derrotado!'+(v.carry?' A chave caiu no chão.':''),220);
        villain=null;vTimer=1800+Math.floor(Math.random()*1800);
      }
    }else if(p.inv<=0&&v.flash===0){
      const k=keyItems.find(k=>k.state==='inv');
      if(k&&!v.carry){
        k.state='stolen';v.carry=k;p.inv=120;p.vy=-8;
        toast(v.name+' roubou sua chave '+KN[k.id]+'! Derrote-o!',220);beep(200,0.3,'sawtooth');
      }else die();
    }
  }
}

// ---------- EASTER EGGS ----------
function spawnEgg(){
  const t=['ufo','pig','gift','ducks','ghost','star'][Math.floor(Math.random()*6)];
  const e={t:0,type:t,dead:false};
  if(t==='ufo'){e.sx=W+60;e.sy=40+Math.random()*50;toast('Eles chegaram... mas só queriam ver o jogo.',200);}
  if(t==='ghost'){e.sx=-60;e.sy=140+Math.random()*60;toast('Um fantasma de jogo antigo passou por aqui...',200);}
  if(t==='star'){e.sx=W*0.4+Math.random()*W*0.5;e.sy=0;toast('Estrela cadente! Faça um pedido.',200);}
  if(t==='ducks'){e.items=Array.from({length:45},()=>({x:Math.random()*W,y:-Math.random()*400,v:1+Math.random()*2}));toast('Chove patinhos de borracha!',200);}
  if(t==='pig'){e.x=camX+W+50;e.y=270+Math.random()*40;e.w=30;e.h=24;toast('Um porco voador! Tente tocar nele.',200);}
  if(t==='gift'){e.x=Math.min(cols*T-200,camX+150+Math.random()*350);e.y=-30;e.w=20;e.h=20;e.vy=0;toast('Caiu um presente do céu!',200);}
  eggs.push(e);
}
function updateEggs(){
  if(--eggTimer<=0){spawnEgg();eggTimer=700+Math.floor(Math.random()*1200);}
  for(const e of eggs){
    e.t++;
    if(e.type==='ufo'){e.sx-=1.6;if(e.sx<-80)e.dead=true;}
    else if(e.type==='ghost'){e.sx+=0.9;e.sy+=Math.sin(e.t/20)*0.8;if(e.sx>W+80)e.dead=true;}
    else if(e.type==='star'){e.sx-=7;e.sy+=3.5;if(e.sy>H)e.dead=true;}
    else if(e.type==='ducks'){for(const d of e.items)d.y+=d.v;if(e.t>420)e.dead=true;}
    else if(e.type==='pig'){
      e.x-=1.4;e.y+=Math.sin(e.t/15)*0.8;
      if(hit(p,e)){e.dead=true;score+=50;toast('Porco voador! Ninguém vai acreditar em você.',200);beep(1000,0.2,'triangle');}
      if(e.x<camX-100)e.dead=true;
    }else if(e.type==='gift'){
      e.vy=Math.min(12,e.vy+0.6);moveY(e);
      if(e.y>H+50)e.dead=true;
      if(hit(p,e)){e.dead=true;score+=100;toast(GIFTS[Math.floor(Math.random()*GIFTS.length)],200);beep(900,0.15,'triangle');}
    }
  }
  eggs=eggs.filter(e=>!e.dead);
}

// ---------- UPDATE ----------
function update(){
  frame++;
  if(toastT>0)toastT--;
  if(msgCool>0)msgCool--;
  if(crown>0)crown--;
  if(tapT>0){tapT--;if(tapT===0)tapN=0;}
  if(state==='title')return;
  if(state==='win'){if(keys.Enter)newGame();return;}
  if(state==='over'){
    if(overT>0)overT--;
    else if(keys.Enter||keys.Space||keys.KeyE)newGame();
    return;
  }
  if(state==='levelup'){
    timer--;
    if(timer<=0){level++;if(level>5)state='win';else loadLevel();}
    return;
  }
  updatePlayer();
  if(state!=='play')return;
  for(const e of enemies){
    if(e.dead){e.t--;continue;}
    e.vy=Math.min(14,e.vy+0.6);
    e.vx=e.dir*(1+level*0.1);
    if(moveX(e))e.dir*=-1;
    moveY(e);
    if(e.onGround){
      const fx=e.dir>0?e.x+e.w+2:e.x-2;
      if(!solid(Math.floor(fx/T),Math.floor((e.y+e.h+4)/T)))e.dir*=-1;
    }
    if(e.y>H+80){e.dead=true;e.t=0;}
    touchEnemy(e);
  }
  enemies=enemies.filter(e=>!e.dead||e.t>0);
  for(const b of bats){
    if(b.dead){b.t--;b.y+=5;continue;}
    b.x=b.bx+Math.sin(frame/50+b.ph)*90;
    b.y=b.y0+Math.sin(frame/20+b.ph)*30;
    touchEnemy(b);
  }
  bats=bats.filter(b=>!b.dead||b.t>0);
  for(const s of spikes)if(p.inv<=0&&hit(p,s))die();
  for(const c of coins){
    if(!c.got&&p.x<c.x+16&&p.x+p.w>c.x&&p.y<c.y+20&&p.y+p.h>c.y){c.got=true;coinsGot++;score+=10;beep(1000,0.1,'triangle');}
  }
  for(const k of keyItems){
    if(k.state!=='world')continue;
    if(k.fall){k.vy=Math.min(12,k.vy+0.6);moveY(k);if(k.y>H+60){k.x=cpX+60;k.y=13*T-40;k.vy=0;}}
    if(hit(p,k)){k.state='inv';toast('Você pegou a chave '+KN[k.id]+'!',150);beep(1200,0.15,'triangle');}
  }
  for(const s of safes){
    if(s.open||!hit(p,{x:s.x-6,y:s.y-6,w:s.w+12,h:s.h+12}))continue;
    const k=keyItems.find(k=>k.id===s.kind&&k.state==='inv');
    if(k){
      k.state='used';s.open=true;burst(s.x+15,s.y,20);beep(600,0.3,'triangle');
      if(s.kind==='gold'){
        keyItems.push({id:'silver',x:s.x+7,y:s.y-24,w:16,h:16,state:'world',fall:true,vy:-4});
        toast('O cofre abriu! Tinha uma chave prateada dentro.',220);
      }else{lives++;score+=300;toast('Cofre azul: +1 vida e 300 pontos!',220);}
    }else if(msgCool<=0){
      toast('Cofre '+(s.kind==='gold'?'dourado':'azul')+' trancado. Falta a chave '+KN[s.kind]+'.',160);msgCool=200;
    }
  }
  if(!gate.open){
    const gx=gate.col*T;
    if(p.x+p.w>gx-10){
      const k=keyItems.find(k=>k.id==='silver'&&k.state==='inv');
      if(k){
        k.state='used';gate.open=true;
        for(let rr=8;rr<=12;rr++)grid[rr][gate.col]=0;
        burst(gx+16,12*T,30);toast('O portão se abriu!',160);beep(500,0.5,'triangle');
      }else if(msgCool<=0){toast('Portão trancado. Precisa da chave prateada.',160);msgCool=200;}
    }
  }
  if(!cp.reached&&p.x>cp.x){cp.reached=true;cpX=cp.x;toast('Ponto de retorno salvo!',120);beep(900,0.2);}
  if(!villain){if(vTimer>0&&--vTimer===0)spawnVillain();}
  else updateVillain();
  updateEggs();
  for(const q of particles){q.x+=q.vx;q.y+=q.vy;q.vy+=0.3;q.life--;}
  particles=particles.filter(q=>q.life>0);
  const poleX=(cols-4)*T+14;
  if(p.x+p.w>poleX&&state==='play'){
    state='levelup';timer=140;score+=500+lives*100;villain=null;beep(880,0.4,'triangle');
  }
  camX=Math.max(0,Math.min(cols*T-W,p.x-W/3));
}

// ---------- DESENHO ----------
function drawBG(){
  const g=ctx.createLinearGradient(0,0,0,H);
  g.addColorStop(0,'#5fb3ff');g.addColorStop(1,'#d6f0ff');
  ctx.fillStyle=g;ctx.fillRect(0,0,W,H);
  ctx.fillStyle='#9ad98a';
  ctx.beginPath();ctx.moveTo(0,H);
  for(let x=0;x<=W;x+=10)ctx.lineTo(x,330+Math.sin((x+camX*0.3)/90)*30+Math.sin((x+camX*0.3)/37)*8);
  ctx.lineTo(W,H);ctx.fill();
  ctx.fillStyle='rgba(255,255,255,0.9)';
  for(let i=0;i<30;i++){
    const wx=i*300+(i*71)%100;
    const x=((wx-camX*0.4)%(W+300)+(W+300))%(W+300)-150;
    const y=40+(i*37)%90;
    ctx.beginPath();ctx.arc(x,y,20,0,7);ctx.arc(x+22,y-8,24,0,7);ctx.arc(x+48,y,20,0,7);ctx.fill();
  }
}
function drawSky(){
  for(const e of eggs){
    if(e.type==='ufo'){
      ctx.fillStyle='rgba(180,255,200,'+(0.15+0.1*Math.sin(e.t/8))+')';
      ctx.beginPath();ctx.moveTo(e.sx-8,e.sy+8);ctx.lineTo(e.sx-40,e.sy+260);ctx.lineTo(e.sx+40,e.sy+260);ctx.lineTo(e.sx+8,e.sy+8);ctx.fill();
      ctx.fillStyle='#aab';ctx.beginPath();ctx.ellipse(e.sx,e.sy,30,9,0,0,7);ctx.fill();
      ctx.fillStyle='#8fd3ff';ctx.beginPath();ctx.arc(e.sx,e.sy-4,12,Math.PI,0);ctx.fill();
      ctx.fillStyle=Math.floor(e.t/6)%2?'#ff0':'#f60';
      for(let i=-1;i<=1;i++){ctx.beginPath();ctx.arc(e.sx+i*14,e.sy+2,2.5,0,7);ctx.fill();}
    }else if(e.type==='ghost'){
      ctx.globalAlpha=0.5;ctx.fillStyle='#fff';
      ctx.beginPath();ctx.arc(e.sx,e.sy,22,Math.PI,0);
      ctx.lineTo(e.sx+22,e.sy+30);
      for(let i=0;i<4;i++)ctx.lineTo(e.sx+22-(i+1)*11,e.sy+(i%2?30:38));
      ctx.fill();
      ctx.fillStyle='#000';ctx.fillRect(e.sx-10,e.sy-5,5,8);ctx.fillRect(e.sx+5,e.sy-5,5,8);
      ctx.globalAlpha=1;
    }else if(e.type==='star'){
      ctx.strokeStyle='#fff';ctx.lineWidth=3;ctx.beginPath();ctx.moveTo(e.sx,e.sy);ctx.lineTo(e.sx+50,e.sy-25);ctx.stroke();
      ctx.fillStyle='#ffd700';ctx.beginPath();ctx.arc(e.sx,e.sy,5,0,7);ctx.fill();
    }else if(e.type==='ducks'){
      for(const d of e.items){
        if(d.y<-10||d.y>H+10)continue;
        ctx.fillStyle='#ffe033';ctx.beginPath();ctx.arc(d.x,d.y,6,0,7);ctx.fill();
        ctx.fillStyle='#ff8c1a';ctx.fillRect(d.x+4,d.y,5,3);
      }
    }
  }
}
function drawWorldEggs(){
  for(const e of eggs){
    if(e.type==='pig'){
      const x=e.x-camX,y=e.y,f=Math.sin(e.t/4)*6;
      ctx.fillStyle='#fff';ctx.beginPath();ctx.ellipse(x+14,y+2+f,10,5,0.4,0,7);ctx.fill();
      ctx.fillStyle='#ffb6c1';ctx.beginPath();ctx.ellipse(x+15,y+14,15,10,0,0,7);ctx.fill();
      ctx.fillStyle='#ff91a4';ctx.beginPath();ctx.ellipse(x+3,y+14,5,4,0,0,7);ctx.fill();
      ctx.fillStyle='#000';ctx.fillRect(x+7,y+9,2,3);
    }else if(e.type==='gift'){
      const x=e.x-camX,y=e.y;
      ctx.fillStyle='#e63946';ctx.fillRect(x,y,20,20);
      ctx.fillStyle='#ffd700';ctx.fillRect(x+8,y,4,20);ctx.fillRect(x,y+8,20,4);
    }
  }
}
function drawTiles(){
  const c0=Math.floor(camX/T),c1=Math.min(cols-1,c0+Math.ceil(W/T)+1);
  for(let r=0;r<ROWS;r++)for(let c=c0;c<=c1;c++){
    const t=grid[r][c];if(!t)continue;
    const x=c*T-camX,y=r*T;
    if(t===4){
      ctx.fillStyle='#5a6270';ctx.fillRect(x,y,T,T);
      ctx.fillStyle='#3a404a';for(let i=0;i<4;i++)ctx.fillRect(x+3+i*8,y,3,T);
      if(r===10){ctx.fillStyle='#ffd700';ctx.beginPath();ctx.arc(x+16,y+16,7,0,7);ctx.fill();ctx.fillStyle='#000';ctx.fillRect(x+14,y+16,4,8);}
    }else if(t===2){
      ctx.fillStyle='#d9822b';ctx.fillRect(x,y,T,T);
      ctx.strokeStyle='#8a4b12';ctx.lineWidth=2;ctx.strokeRect(x+1,y+1,T-2,T-2);
      ctx.beginPath();ctx.moveTo(x,y+T/2);ctx.lineTo(x+T,y+T/2);ctx.moveTo(x+T/2,y);ctx.lineTo(x+T/2,y+T/2);ctx.stroke();
    }else{
      ctx.fillStyle='#8b5a2b';ctx.fillRect(x,y,T,T);
      ctx.fillStyle='#74481f';ctx.fillRect(x+6,y+10,5,5);ctx.fillRect(x+20,y+20,5,5);
      if(t===1){ctx.fillStyle='#4cc04c';ctx.fillRect(x,y,T,9);ctx.fillStyle='#2e9a2e';ctx.fillRect(x,y+8,T,2);}
    }
  }
}
function drawCoins(){
  for(const c of coins){
    if(c.got)continue;
    const w=Math.abs(Math.cos((frame+c.x)/12))*7+2;
    ctx.fillStyle='#ffd700';ctx.strokeStyle='#b8860b';ctx.lineWidth=2;
    ctx.beginPath();ctx.ellipse(c.x+8-camX,c.y+10,w,9,0,0,7);ctx.fill();ctx.stroke();
  }
}
function drawSpikes(){
  ctx.fillStyle='#b0b4bb';ctx.strokeStyle='#555';ctx.lineWidth=1.5;
  for(const s of spikes){
    const x=s.x-camX;if(x<-40||x>W+40)continue;
    for(let i=0;i<3;i++){
      ctx.beginPath();ctx.moveTo(x+i*(s.w/3),s.y+s.h);ctx.lineTo(x+i*(s.w/3)+s.w/6,s.y);ctx.lineTo(x+(i+1)*(s.w/3),s.y+s.h);ctx.fill();ctx.stroke();
    }
  }
}
function drawSafe(s){
  const x=s.x-camX,y=s.y;
  ctx.fillStyle='#4a4f57';ctx.fillRect(x,y,s.w,s.h);
  ctx.fillStyle=KC[s.kind];ctx.fillRect(x,y,s.w,5);
  ctx.strokeStyle='#222';ctx.lineWidth=2;ctx.strokeRect(x+1,y+1,s.w-2,s.h-2);
  if(s.open){ctx.fillStyle='#222';ctx.fillRect(x+4,y+8,s.w-8,s.h-12);ctx.fillStyle='#6b717a';ctx.fillRect(x+s.w-4,y+8,8,s.h-12);}
  else{ctx.fillStyle='#bbb';ctx.beginPath();ctx.arc(x+15,y+17,7,0,7);ctx.fill();ctx.fillStyle='#222';ctx.fillRect(x+14,y+12,2,6);}
}
function drawKeyShape(x,y,id){
  ctx.fillStyle=KC[id];ctx.strokeStyle='#333';ctx.lineWidth=1.5;
  ctx.beginPath();ctx.arc(x+4,y+8,5,0,7);ctx.fill();ctx.stroke();
  ctx.fillRect(x+8,y+7,9,3);ctx.fillRect(x+13,y+9,2,4);ctx.fillRect(x+10,y+9,2,3);
}
function drawPony(walk){
  ctx.fillStyle='#c98a4b';
  ctx.fillRect(-14+walk,-8,4,8);ctx.fillRect(-7-walk,-8,4,8);ctx.fillRect(5+walk,-8,4,8);ctx.fillRect(12-walk,-8,4,8);
  ctx.beginPath();ctx.ellipse(0,-14,18,9,0,0,7);ctx.fill();
  ctx.beginPath();ctx.ellipse(17,-22,8,6,-0.5,0,7);ctx.fill();
  ctx.fillStyle='#5a3a1a';
  ctx.fillRect(10,-30,4,10);ctx.beginPath();ctx.moveTo(-17,-16);ctx.lineTo(-26,-6);ctx.lineTo(-16,-10);ctx.fill();
  ctx.fillStyle='#000';ctx.fillRect(19,-25,2,2);
}
function drawDecor(){
  for(const d of decor){
    if(d.ridden)continue;
    const x=d.x-camX,y=d.y;if(x<-60||x>W+60)continue;
    if(d.kind==='pony'){ctx.save();ctx.translate(x+20,y+d.h);drawPony(Math.sin(frame/12)*1);ctx.restore();}
    else if(d.kind==='bench'){
      ctx.fillStyle='#8a5a2b';ctx.fillRect(x,y+4,d.w,5);ctx.fillRect(x,y-6,d.w,4);ctx.fillRect(x+4,y+9,4,11);ctx.fillRect(x+d.w-8,y+9,4,11);
    }else if(d.kind==='lever'){
      ctx.fillStyle='#555';ctx.fillRect(x,y+22,20,8);
      ctx.strokeStyle='#c33';ctx.lineWidth=4;ctx.beginPath();ctx.moveTo(x+10,y+24);ctx.lineTo(x+10+(d.on?12:-12),y+6);ctx.stroke();
      ctx.fillStyle='#c33';ctx.beginPath();ctx.arc(x+10+(d.on?12:-12),y+6,5,0,7);ctx.fill();
    }else if(d.kind==='sign'){
      ctx.fillStyle='#6b4a22';ctx.fillRect(x+13,y+14,4,22);
      ctx.fillStyle='#d9b26b';ctx.fillRect(x,y,30,16);ctx.strokeStyle='#6b4a22';ctx.lineWidth=2;ctx.strokeRect(x,y,30,16);
      ctx.fillStyle='#000';ctx.font='bold 12px sans-serif';ctx.textAlign='center';ctx.fillText('?',x+15,y+13);
    }
  }
}
function drawCheckpoint(){
  const x=cp.x+14-camX;
  ctx.fillStyle='#eee';ctx.fillRect(x,13*T-60,4,60);
  ctx.fillStyle=cp.reached?'#2ecc71':'#bbb';
  ctx.beginPath();ctx.moveTo(x+4,13*T-60);ctx.lineTo(x+34,13*T-50);ctx.lineTo(x+4,13*T-40);ctx.fill();
}
function drawFlag(){
  const x=(cols-4)*T+14-camX;
  ctx.fillStyle='#eee';ctx.fillRect(x,6*T,5,7*T);
  ctx.fillStyle='#e63946';
  ctx.beginPath();ctx.moveTo(x+5,6*T);ctx.lineTo(x+50,6*T+16+Math.sin(frame/8)*3);ctx.lineTo(x+5,6*T+32);ctx.fill();
  ctx.fillStyle='#ffd700';ctx.beginPath();ctx.arc(x+2,6*T-4,6,0,7);ctx.fill();
}
function drawEnemy(e){
  const x=e.x-camX,y=e.y;
  if(x<-40||x>W+40)return;
  if(e.dead){ctx.fillStyle='#7a3b8f';ctx.fillRect(x,y+e.h-6,e.w,6);return;}
  ctx.fillStyle='#8e44ad';
  ctx.beginPath();ctx.ellipse(x+e.w/2,y+e.h/2,e.w/2,e.h/2,0,0,7);ctx.fill();
  ctx.fillStyle='#5b2c6f';
  const s=Math.sin(frame/5)*3;
  ctx.fillRect(x+2+s,y+e.h-4,8,4);ctx.fillRect(x+e.w-10-s,y+e.h-4,8,4);
  ctx.fillStyle='#fff';ctx.fillRect(x+5,y+6,6,7);ctx.fillRect(x+15,y+6,6,7);
  ctx.fillStyle='#000';ctx.fillRect(x+7+e.dir*2,y+8,3,4);ctx.fillRect(x+17+e.dir*2,y+8,3,4);
  ctx.strokeStyle='#000';ctx.lineWidth=2;
  ctx.beginPath();ctx.moveTo(x+4,y+4);ctx.lineTo(x+11,y+7);ctx.moveTo(x+22,y+4);ctx.lineTo(x+15,y+7);ctx.stroke();
}
function drawBat(e){
  const x=e.x-camX,y=e.y;if(x<-50||x>W+50)return;
  const f=e.dead?0:Math.sin(frame/4)*8;
  ctx.fillStyle='#2b2b3a';
  ctx.beginPath();ctx.ellipse(x+14,y+12,8,9,0,0,7);ctx.fill();
  ctx.beginPath();ctx.moveTo(x+8,y+10);ctx.lineTo(x-8,y+2+f);ctx.lineTo(x+6,y+16);ctx.fill();
  ctx.beginPath();ctx.moveTo(x+20,y+10);ctx.lineTo(x+36,y+2+f);ctx.lineTo(x+22,y+16);ctx.fill();
  ctx.fillStyle='#f33';ctx.fillRect(x+9,y+8,3,3);ctx.fillRect(x+16,y+8,3,3);
}
function drawVillain(){
  const v=villain;if(!v)return;
  if(v.flash>0&&Math.floor(frame/3)%2)return;
  const x=v.x-camX,y=v.y,f=Math.sin(frame/3)*10;
  ctx.fillStyle='#15151f';
  ctx.beginPath();ctx.moveTo(x+10,y+16);ctx.lineTo(x-22,y-6+f);ctx.lineTo(x-4,y+30);ctx.fill();
  ctx.beginPath();ctx.moveTo(x+34,y+16);ctx.lineTo(x+66,y-6+f);ctx.lineTo(x+48,y+30);ctx.fill();
  ctx.beginPath();ctx.ellipse(x+22,y+22,20,18,0,0,7);ctx.fill();
  ctx.fillStyle='#ff8c1a';
  const bx=v.dir>0?36:8,bd=v.dir>0?16:-16;
  ctx.beginPath();ctx.moveTo(x+bx,y+22);ctx.lineTo(x+bx+bd,y+28);ctx.lineTo(x+bx,y+32);ctx.fill();
  ctx.fillStyle='#f22';ctx.fillRect(x+12,y+14,6,6);ctx.fillRect(x+26,y+14,6,6);
  ctx.fillStyle='#000';ctx.fillRect(x+8,y+4,28,5);ctx.fillRect(x+14,y-8,16,13);
  for(let i=0;i<v.hp;i++){ctx.fillStyle='#e63946';ctx.fillRect(x+4+i*14,y-22,10,6);}
  ctx.fillStyle='#fff';ctx.font='bold 12px sans-serif';ctx.textAlign='center';ctx.fillText(v.name,x+22,y-28);
}
function drawDuck(){
  if(p.inv>0&&Math.floor(frame/4)%2===0)return;
  ctx.save();
  ctx.translate(p.x-camX+p.w/2,p.y+p.h);
  ctx.scale(p.dir,1);
  const walk=p.onGround&&Math.abs(p.vx)>0.5?Math.sin(frame/3)*4:0;
  if(riding){drawPony(walk);ctx.translate(0,-14);}
  if(sitting)ctx.scale(1,0.85);
  const col=rainbow?'hsl('+(frame*6%360)+',90%,55%)':'#ffe033';
  ctx.fillStyle='#ff8c1a';
  ctx.fillRect(-9+walk,-5,9,5);ctx.fillRect(1-walk,-5,9,5);
  ctx.fillStyle=col;ctx.beginPath();ctx.ellipse(-1,-14,12,10,0,0,7);ctx.fill();
  ctx.fillStyle='rgba(0,0,0,0.15)';ctx.beginPath();ctx.ellipse(-5,-13,6,5,0.3,0,7);ctx.fill();
  ctx.fillStyle=col;ctx.beginPath();ctx.arc(5,-24,8,0,7);ctx.fill();
  ctx.fillStyle='#ff8c1a';ctx.beginPath();ctx.ellipse(14,-22,6,3,0,0,7);ctx.fill();
  ctx.fillStyle='#fff';ctx.beginPath();ctx.arc(7,-26,3,0,7);ctx.fill();
  ctx.fillStyle='#000';ctx.beginPath();ctx.arc(8,-26,1.5,0,7);ctx.fill();
  ctx.fillStyle='#e63946';ctx.beginPath();ctx.arc(4,-28,7,Math.PI,0);ctx.fill();ctx.fillRect(4,-29,10,3);
  if(crown>0){
    ctx.fillStyle='#ffd700';ctx.beginPath();
    ctx.moveTo(-2,-33);ctx.lineTo(-2,-42);ctx.lineTo(2,-37);ctx.lineTo(6,-44);ctx.lineTo(10,-37);ctx.lineTo(14,-42);ctx.lineTo(14,-33);ctx.fill();
  }
  ctx.restore();
}
function drawHUD(){
  ctx.fillStyle='rgba(0,0,0,0.5)';ctx.fillRect(0,0,W,30);
  ctx.fillStyle='#fff';ctx.font='bold 14px sans-serif';ctx.textAlign='left';
  ctx.fillText('\u2665 '+Math.max(0,lives),10,20);
  ctx.fillText('Moedas '+coinsGot,60,20);
  ctx.fillText('Pontos '+score,160,20);
  ctx.fillText('Fase '+level+'/5',270,20);
  let kx=350;
  for(const id of ['gold','blue','silver'])if(hasKey(id)){drawKeyShape(kx,6,id);kx+=26;}
  const prog=Math.min(1,p.x/(cols*T));
  ctx.fillStyle='#555';ctx.fillRect(500,10,200,10);
  ctx.fillStyle='#4cc04c';ctx.fillRect(500,10,200*prog,10);
}
function drawToast(){
  if(toastT<=0)return;
  ctx.font='bold 15px sans-serif';ctx.textAlign='center';
  const w=Math.min(W-20,ctx.measureText(toastMsg).width+30);
  ctx.globalAlpha=Math.min(1,toastT/20);
  ctx.fillStyle='rgba(0,0,0,0.75)';ctx.fillRect(W/2-w/2,40,w,30);
  ctx.fillStyle='#fff';ctx.fillText(toastMsg,W/2,60,W-30);
  ctx.globalAlpha=1;
}
function center(t,y,size,color){
  ctx.textAlign='center';ctx.font='bold '+size+'px sans-serif';
  ctx.fillStyle='rgba(0,0,0,0.6)';ctx.fillText(t,W/2+2,y+2);
  ctx.fillStyle=color;ctx.fillText(t,W/2,y);
}
function draw(){
  drawBG();
  if(state==='title'){
    center('PATO AVENTUREIRO',150,52,'#ffd700');
    center('Ache a chave, abra o cofre, destrave o portão.',200,20,'#fff');
    center('Cuidado: nem tudo avisa quando vai aparecer.',232,18,'#fff');
    center('Toque na tela ou aperte uma tecla para começar',320,20,Math.floor(frame/30)%2?'#fff':'#ffd700');
    ctx.save();ctx.translate(W/2-20,430);ctx.scale(2,2);ctx.restore();
    return;
  }
  drawSky();drawTiles();drawCheckpoint();drawSpikes();drawDecor();
  for(const s of safes)drawSafe(s);
  for(const k of keyItems)if(k.state==='world'){const y=k.y+(k.fall?0:Math.sin(frame/15)*3);drawKeyShape(k.x-camX,y,k.id);}
  drawCoins();drawFlag();drawWorldEggs();
  for(const e of enemies)drawEnemy(e);
  for(const b of bats)drawBat(b);
  drawDuck();
  for(const q of particles){ctx.fillStyle=q.c;ctx.fillRect(q.x-camX,q.y,4,4);}
  if(villain){ctx.fillStyle='rgba(40,0,50,0.22)';ctx.fillRect(0,0,W,H);drawVillain();}
  if(state==='play'){
    if(riding){bubble('AÇÃO: descer');}
    else if(sitting){bubble('AÇÃO: levantar');}
    else if(nearDecor())bubble('toque AÇÃO');
  }
  drawHUD();drawToast();
  if(state==='levelup')center('FASE COMPLETA!',230,48,'#ffd700');
  if(state==='over'){
    ctx.fillStyle='rgba(0,0,0,0.6)';ctx.fillRect(0,0,W,H);
    center('GAME OVER',210,56,'#ff4d4d');
    center('Pontos: '+score+'  |  Moedas: '+coinsGot,260,24,'#fff');
    if(overT<=0)center('Toque na tela (ou ENTER) para jogar de novo',310,20,'#fff');
  }
  if(state==='win'){
    ctx.fillStyle='rgba(0,0,0,0.65)';ctx.fillRect(0,0,W,H);
    center('VOCÊ ZEROU O JOGO!',200,48,'#ffd700');
    center('Pontos finais: '+score,255,26,'#fff');
    center('Toque na tela para recomeçar',310,20,'#fff');
  }
}
function bubble(t){
  const x=p.x-camX+p.w/2,y=p.y-52;
  ctx.font='bold 12px sans-serif';ctx.textAlign='center';
  const w=ctx.measureText(t).width+14;
  ctx.fillStyle='rgba(255,255,255,0.9)';ctx.fillRect(x-w/2,y-14,w,20);
  ctx.fillStyle='#000';ctx.fillText(t,x,y);
}

let last=performance.now(),acc=0;
function loop(t){
  acc+=Math.min(100,t-last);last=t;
  while(acc>=1000/60){update();acc-=1000/60;}
  draw();requestAnimationFrame(loop);
}
requestAnimationFrame(loop);
})(); daddadwwdadad
</script>
'''
display(HTML(html))
open('jogo_pato.html','w',encoding='utf-8').write(html)  # cópia salva, caso queira baixar

32787